<a href="https://colab.research.google.com/github/x1001000/Colab-Notebooks/blob/main/%E4%BD%BF%E7%94%A8%E8%81%AF%E7%99%BC%E7%A7%91%E9%96%8B%E6%BA%90%E6%A8%A1%E5%9E%8BBreeze_ASR_25%E7%94%9F%E6%88%90SRT%E5%AD%97%E5%B9%95%E6%AA%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 上傳MP3：左側面板 > 檔案 > 上傳按鈕

In [18]:
import glob
mp3_files = glob.glob('*.mp3')
if mp3_files:
    if len(mp3_files) > 1:
        for i, mp3_file in enumerate(mp3_files):
            print(f'#{i+1} {mp3_file}')
        mp3_filename = mp3_files[int(input('選一個MP3 #'))-1]
    else:
        mp3_filename = mp3_files[0]
    print(f'完成上傳MP3 👉 {mp3_filename}')
else:
    print("請先上傳MP3，然後再執行一次")

完成上傳MP3 👉 250904_podcast-v2.mp3


# 下載模型、安裝環境、生成文字及時間戳（耗時約半小時但無API費用）

In [ ]:
import torchaudio
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutomaticSpeechRecognitionPipeline

# 1. Load audio
audio_path = mp3_filename
waveform, sample_rate = torchaudio.load(audio_path)

# 2. Preprocess
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)
waveform = waveform.squeeze().numpy()

if sample_rate != 16_000:
    resampler = torchaudio.transforms.Resample(sample_rate, 16_000)
    waveform = resampler(torch.tensor(waveform)).numpy()
    sample_rate = 16_000

# 3. Load Model
processor = WhisperProcessor.from_pretrained("MediaTek-Research/Breeze-ASR-25")
model = WhisperForConditionalGeneration.from_pretrained("MediaTek-Research/Breeze-ASR-25").to("cuda").eval()

# 4. Build Pipeline
asr_pipeline = AutomaticSpeechRecognitionPipeline(
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=0
)

# 6. Inference
output = asr_pipeline(waveform, return_timestamps=True)
print("Result:", output["text"])

# 下載SRT

In [17]:
def format_timestamp(seconds):
    """Converts seconds to SRT timestamp format (HH:MM:SS,ms)."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = seconds % 60
    milliseconds = int((seconds - int(seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{int(seconds):02},{milliseconds:03}"

srt_content = ""
for i, chunk in enumerate(output['chunks']):
    start_time = chunk['timestamp'][0]
    end_time = chunk['timestamp'][1]
    text = chunk['text']

    # SRT format:
    # Subtitle number
    # Start time --> End time
    # Text
    # Blank line

    srt_content += f"{i + 1}\n"
    srt_content += f"{format_timestamp(start_time)} --> {format_timestamp(end_time)}\n"
    srt_content += f"{text}\n\n"

# print(srt_content)

with open(f"{mp3_filename}.srt", "w") as f:
    f.write(srt_content)

from google.colab import files
files.download(f"{mp3_filename}.srt")
print(f"完成下載SRT 👉 {mp3_filename}.srt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

完成下載SRT 👉 250904_podcast-v2.mp3.srt
